# openai_whisper

## Example 1: the tokenizers

In [ ]:
# Example 1: the multilingual tokenizer round-trips text and knows the language and task tokens
import whisper
from whisper.tokenizer import LANGUAGES, get_tokenizer

tokenizer = get_tokenizer(multilingual=True, language="en", task="transcribe")
ids = tokenizer.encode(" Hello world, this is Whisper.")
assert tokenizer.decode(ids) == " Hello world, this is Whisper."
assert tokenizer.sot_sequence[0] == tokenizer.sot and len(tokenizer.sot_sequence) == 3
assert "en" in LANGUAGES and len(LANGUAGES) >= 99
assert "tiny" in whisper.available_models() and "large-v3" in whisper.available_models()
print("Example 1:", len(ids), "tokens;", tokenizer.decode(tokenizer.sot_sequence))

In [ ]:
# Example 2: load_audio runs ffmpeg (16 kHz mono float32); the log-mel spectrogram has 100 frames per second
import pathlib, subprocess, tempfile
import whisper

wav = pathlib.Path(tempfile.mkdtemp()) / "tone.wav"
subprocess.run(["ffmpeg", "-v", "error", "-f", "lavfi", "-i", "sine=frequency=440:sample_rate=44100:duration=1",
                "-ac", "2", str(wav)], check=True)
audio = whisper.load_audio(str(wav))
assert audio.dtype.name == "float32" and audio.shape == (16000,), (audio.dtype, audio.shape)
mel = whisper.log_mel_spectrogram(whisper.pad_or_trim(audio))
assert tuple(mel.shape) == (80, 3000), mel.shape  # 30 s window, 80 mel bands
print("Example 2:", audio.shape, tuple(mel.shape), float(mel.max()))

## Example 3: a tiny random Whisper model, end to end on the CPU

In [ ]:
# Example 3: build a 1-layer model from ModelDimensions (no download) and run the real decoding loop
import torch
import whisper
from whisper.model import ModelDimensions, Whisper

torch.manual_seed(0)
dims = ModelDimensions(n_mels=80, n_audio_ctx=1500, n_audio_state=64, n_audio_head=2, n_audio_layer=1,
                       n_vocab=51865, n_text_ctx=448, n_text_state=64, n_text_head=2, n_text_layer=1)
model = Whisper(dims).eval()
mel = torch.zeros(80, 3000)
features = model.embed_audio(mel.unsqueeze(0))
assert tuple(features.shape) == (1, 1500, 64), features.shape
result = whisper.decode(model, mel, whisper.DecodingOptions(language="en", fp16=False, sample_len=4,
                                                            without_timestamps=True))
assert isinstance(result.text, str) and len(result.tokens) <= 4, result
print("Example 3:", tuple(features.shape), "decoded", len(result.tokens), "tokens from random weights")

## Example 4: the text normalizers

In [ ]:
# Example 4: EnglishTextNormalizer (what MLPerf's Whisper accuracy script scores with) and BasicTextNormalizer
from whisper.normalizers import BasicTextNormalizer, EnglishTextNormalizer

english = EnglishTextNormalizer()
assert english("Mr. Smith paid $20.50 on the 3rd of May!") == english("mister smith paid 20.50 dollars on the third of may")
assert english("I'm gonna colour it") == "i am going to color it", english("I'm gonna colour it")
basic = BasicTextNormalizer()
assert basic("Héllo, World! (aside)") == "héllo world ", repr(basic("Héllo, World! (aside)"))
print("Example 4:", english("Mr. Smith paid $20.50 on the 3rd of May!"))

## Example 5: dynamic time warping for word timestamps

In [ ]:
# Example 5: timing.dtw (numba-compiled on the CPU) aligns a cost matrix; median_filter smooths attention weights
import numpy as np
import torch
from whisper.timing import dtw, median_filter

cost = np.full((4, 6), 9.0, dtype=np.float32)
for i, j in [(0, 0), (1, 1), (1, 2), (2, 3), (3, 4), (3, 5)]:
    cost[i, j] = 0.0
text_indices, time_indices = dtw(torch.from_numpy(cost))
assert list(text_indices) == [0, 1, 1, 2, 3, 3] and list(time_indices) == [0, 1, 2, 3, 4, 5], (text_indices, time_indices)
smoothed = median_filter(torch.tensor([[[0.0, 0.0, 9.0, 0.0, 0.0]]]), filter_width=3)  # one spike, reflect-padded
assert smoothed.flatten().tolist() == [0.0, 0.0, 0.0, 0.0, 0.0], smoothed
print("Example 5:", list(zip(text_indices.tolist(), time_indices.tolist())))